# Stage 1 -- C/C++ Dataset Statistics

Produces `outputs/stage1_c_cpp_stats.xlsx` with sheets: **C_Real**, **C_Synth**, **C_AI**, **Filters**, **Legend**.

**Prerequisites:** run `00_download_datasets.ipynb` first.

In [1]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW      = ROOT / 'data' / 'raw'
CWE_XML  = ROOT / 'data' / 'cwec_latest.xml'
OUT_DIR  = ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
XLSX_PATH = OUT_DIR / 'stage1_c_cpp_stats.xlsx'

DATASET_FILTERS = {
    'PrimeVul':      {'branch': 'real',  'language': 'C/C++', 'source': 'HuggingFace benjis/primevul', 'positives': 'target=1, CWE non-empty, single-function commit', 'negatives': 'all target=0 (explicit)', 'cwe_source': 'cwe column (stringified list)', 'filters_applied': 'NVD placeholder drop, CWE split, single-function commit', 'safe_type': 'generic-commit', 'label_quality': '3/5', 'notes': '~60% label accuracy; DiverseVul excluded (dominated)'},
    'ICVul':         {'branch': 'real',  'language': 'C/C++', 'source': 'Google Drive (authors)', 'positives': 'before_change=True, fc_hash in cve_fc_vcc_mapping', 'negatives': 'none', 'cwe_source': 'cve_fc_vcc_mapping.cwe_id', 'filters_applied': 'NVD placeholder drop, CWE non-empty', 'safe_type': 'none', 'label_quality': '5/5', 'notes': 'SZZ VCC attribution; join on fc_hash not vcc_hash'},
    'CVEfixes(C)':   {'branch': 'real',  'language': 'C/C++', 'source': 'HuggingFace hitoshura25/cvefixes', 'positives': 'language in {C,C++}, vulnerable_code non-empty, CWE non-empty, single-function commit', 'negatives': 'none', 'cwe_source': 'cwe_id column (NVD join in source)', 'filters_applied': 'language filter, NVD placeholder drop, single-function commit', 'safe_type': 'none', 'label_quality': '3/5', 'notes': 'NVD CWE can be broad (e.g. CWE-119)'},
    'MegaVul':       {'branch': 'real',  'language': 'C/C++', 'source': 'HuggingFace hitoshura25/megavul', 'positives': 'vulnerable_code non-empty, CWE non-empty, single-function commit', 'negatives': 'none', 'cwe_source': 'cwe_id column', 'filters_applied': 'NVD placeholder drop, CWE split, single-function commit', 'safe_type': 'none', 'label_quality': '3/5', 'notes': 'optional CVSS threshold (not used here)'},
    'SecVulEval':    {'branch': 'real',  'language': 'C/C++', 'source': 'HuggingFace arag0rn/SecVulEval', 'positives': 'is_vulnerable=True, CWE in cwe_list', 'negatives': 'all is_vulnerable=False (explicit)', 'cwe_source': 'cwe_list column (inline)', 'filters_applied': 'NVD placeholder drop, CWE split', 'safe_type': 'pure', 'label_quality': '4/5', 'notes': 'statement-level annotation; fine-grained labels'},
    'CrossVul(C)':   {'branch': 'real',  'language': 'C/C++', 'source': 'Zenodo crossvul.zip', 'positives': 'bad_* files (vulnerable functions)', 'negatives': 'good_* files (fixed functions)', 'cwe_source': 'CWE-NNN directory name', 'filters_applied': 'c/cc/cpp/cxx folder filter', 'safe_type': 'fix-paired', 'label_quality': '3/5', 'notes': ''},
    'SVEN(C)':       {'branch': 'real',  'language': 'C/C++', 'source': 'HuggingFace bstee615/sven', 'positives': 'func_src_before, CWE from vul_type', 'negatives': 'func_src_after (fix-paired safe)', 'cwe_source': 'vul_type column (cwe-NNN format, normalised)', 'filters_applied': 'language==C/C++ (from file extension), empty vul_type dropped', 'safe_type': 'fix-paired', 'label_quality': '4/5', 'notes': ''},
    'Juliet(C)':     {'branch': 'synth', 'language': 'C/C++', 'source': 'NIST SARD juliet-test-suite-for-c-cplusplus-v1-3.zip', 'positives': '*_bad.c files', 'negatives': '*_good*.c files', 'cwe_source': 'directory name (CWE-NNN prefix)', 'filters_applied': 'filename pattern (_bad/_good)', 'safe_type': 'pure', 'label_quality': '5/5', 'notes': 'synthetic; whole file treated as sample'},
    'CASTLE':        {'branch': 'synth', 'language': 'C/C++', 'source': 'GitHub CASTLE-Benchmark/CASTLE-Benchmark', 'positives': 'vulnerable=True, CWE non-empty', 'negatives': 'vulnerable=False', 'cwe_source': 'cwe field in JSON (integer, normalised to CWE-NNN)', 'filters_applied': 'NVD placeholder drop', 'safe_type': 'pure', 'label_quality': '5/5', 'notes': ''},
    'LLMSecEval(C)': {'branch': 'ai',   'language': 'C/C++', 'source': 'Zenodo 5225651 (copilot-cwe-scenarios-dataset)', 'positives': 'gen_scenario/*.c (non-reject Copilot completions)', 'negatives': 'none', 'cwe_source': 'cwe-NNN directory name', 'filters_applied': '.c extension, .reject excluded by glob', 'safe_type': 'none', 'label_quality': '3/5', 'notes': 'Copilot-generated; ~40% actually vulnerable per paper; all labeled as vuln'},
}

print(f'Root: {ROOT}')
print(f'Output: {XLSX_PATH}')

Root: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST
Output: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\outputs\stage1_c_cpp_stats.xlsx


## 1. CWE XML -- download if missing

In [2]:
import io, urllib.request, zipfile

CWE_ZIP_URL = 'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'

if CWE_XML.exists():
    print(f'CWE XML present ({CWE_XML.stat().st_size / 1e6:.1f} MB) -- skipping download.')
else:
    print('Downloading CWE XML from MITRE ...')
    with urllib.request.urlopen(CWE_ZIP_URL, timeout=60) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        xml_name = next(n for n in zf.namelist() if n.endswith('.xml'))
        CWE_XML.write_bytes(zf.read(xml_name))
    print(f'Saved: {CWE_XML} ({CWE_XML.stat().st_size / 1e6:.1f} MB)')

CWE XML present (16.1 MB) -- skipping download.


## 2. Load CWE Navigator

In [3]:
from ingestion.cwe_navigator import CWENavigator

nav = CWENavigator(str(CWE_XML))
print(f'Loaded: {len(nav.weaknesses):,} weaknesses, {len(nav.categories):,} categories')

_parent_ids: set[str] = {p for parents in nav.child_of.values() for p in parents}

def _strip(cwe_str: str) -> str:
    return cwe_str.removeprefix('CWE-').removeprefix('cwe-').strip()

def cwe_label(cwe_str: str) -> str:
    num = _strip(cwe_str)
    if num in nav.categories or num in nav.views:
        return 'category'
    if num not in nav.weaknesses:
        return 'unknown'
    if nav.get_element_name(num).startswith('DEPRECATED:'):
        return 'deprecated'
    return 'leaf' if num not in _parent_ids else 'non-leaf'

Loaded: 969 weaknesses, 420 categories


## 3. Extract samples (skip if files missing)

In [4]:
from collections import Counter
from ingestion.schema     import FunctionSample
from ingestion.primevul   import extract_primevul
from ingestion.icvul      import extract_icvul
from ingestion.cvefixes   import extract_cvefixes
from ingestion.megavul    import extract_megavul
from ingestion.secvuleval import extract_secvuleval
from ingestion.crossvul   import extract_crossvul
from ingestion.sven       import extract_sven
from ingestion.juliet     import extract_juliet
from ingestion.castle     import extract_castle
from ingestion.llmseceval import extract_llmseceval

def _try(name, loader):
    try:
        s = loader()
        print(f'  {name:20s}: {len(s):>7,} samples')
        return s
    except FileNotFoundError as e:
        print(f'  {name:20s}: SKIPPED -- {e}')
        return []

collections: dict[str, list[FunctionSample]] = {}

collections['PrimeVul']      = _try('PrimeVul',      lambda: extract_primevul(RAW/'primevul_train.jsonl') + extract_primevul(RAW/'primevul_test.jsonl'))
collections['ICVul']         = _try('ICVul',         lambda: extract_icvul(next((RAW/'icvul').rglob('function_info.csv')).parent) if (RAW/'icvul').exists() else (_ for _ in ()).throw(FileNotFoundError('icvul/ not found')))
collections['CVEfixes(C)']   = _try('CVEfixes(C)',   lambda: extract_cvefixes(RAW/'cvefixes', language='C'))
collections['MegaVul']       = _try('MegaVul',       lambda: extract_megavul(RAW/'megavul'))
collections['SecVulEval']    = _try('SecVulEval',    lambda: extract_secvuleval(RAW/'secvuleval.csv'))
collections['CrossVul(C)']   = _try('CrossVul(C)',   lambda: extract_crossvul(RAW/'crossvul.zip', language='C/C++'))
collections['SVEN(C)']       = _try('SVEN(C)',       lambda: extract_sven(RAW/'sven', language='C/C++'))
collections['Juliet(C)']     = _try('Juliet(C)',     lambda: extract_juliet(RAW/'juliet_c.zip', language='C/C++'))
collections['CASTLE']        = _try('CASTLE',        lambda: extract_castle(RAW/'castle'/'datasets'))
collections['LLMSecEval(C)'] = _try('LLMSecEval(C)', lambda: extract_llmseceval(RAW/'llmseceval', language='C/C++'))

collections = {k: v for k, v in collections.items() if v}
print(f'\nLoaded: {list(collections.keys())}')

  PrimeVul            : 200,008 samples
  ICVul               :   5,782 samples
  CVEfixes(C)         :   2,518 samples
  MegaVul             :   1,130 samples
  SecVulEval          :  24,723 samples
  CrossVul(C)         :   8,072 samples
  SVEN(C)             :     846 samples
  Juliet(C)           :  10,214 samples
  CASTLE              :     250 samples
  LLMSecEval(C)       :     516 samples

Loaded: ['PrimeVul', 'ICVul', 'CVEfixes(C)', 'MegaVul', 'SecVulEval', 'CrossVul(C)', 'SVEN(C)', 'Juliet(C)', 'CASTLE', 'LLMSecEval(C)']


## 4. Compute statistics

In [5]:
stats: dict[str, dict] = {}

for name, samples in collections.items():
    vuln    = [s for s in samples if s.label == 1]
    safe    = [s for s in samples if s.label == 0]
    counter = Counter(cwe for s in vuln for cwe in s.cwes)
    branch  = samples[0].branch if samples else ''
    stats[name] = {
        'total': len(samples), 'vulnerable': len(vuln), 'safe': len(safe),
        'unique_cwes': len(counter), 'cwe_counts': counter, 'branch': branch,
    }

print(f'{"Dataset":<22} {"Branch":>6} {"Total":>8} {"Vuln":>8} {"Safe":>8} {"CWEs":>6}')
print('-' * 64)
for n, s in stats.items():
    print(f'{n:<22} {s["branch"]:>6} {s["total"]:>8,} {s["vulnerable"]:>8,} {s["safe"]:>8,} {s["unique_cwes"]:>6,}')

Dataset                Branch    Total     Vuln     Safe   CWEs
----------------------------------------------------------------
PrimeVul                 real  200,008    4,834  195,174    121
ICVul                    real    5,782    5,782        0    127
CVEfixes(C)              real    2,518    2,518        0    117
MegaVul                  real    1,130    1,130        0     75
SecVulEval               real   24,723   10,281   14,442    131
CrossVul(C)              real    8,072    4,036    4,036    104
SVEN(C)                  real      846      423      423      8
Juliet(C)               synth   10,214    4,323    5,891     52
CASTLE                  synth      250      150      100     25
LLMSecEval(C)              ai      516      516        0     11


## 5. Build Excel report

In [6]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FILL_HEADER     = PatternFill('solid', fgColor='1F4E79')
FILL_LEAF       = PatternFill('solid', fgColor='92D050')
FILL_NONLEAF    = PatternFill('solid', fgColor='FFC000')
FILL_UNKNOWN    = PatternFill('solid', fgColor='BFBFBF')
FILL_DEPRECATED = PatternFill('solid', fgColor='FFB3B3')
FILL_CATEGORY   = PatternFill('solid', fgColor='D2B4DE')
FILL_ROW_ODD    = PatternFill('solid', fgColor='F2F2F2')
FONT_HEADER     = Font(bold=True, color='FFFFFF', name='Calibri', size=11)
FONT_CWE        = Font(bold=True, color='000000', name='Calibri', size=10)
FONT_DATA       = Font(name='Calibri', size=10)
FONT_DATA_B     = Font(bold=True, name='Calibri', size=10)
ALIGN_C = Alignment(horizontal='center', vertical='center', wrap_text=True)
ALIGN_L = Alignment(horizontal='left', vertical='center')
THIN    = Side(style='thin', color='D3D3D3')
BORDER  = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

_CWE_FILL = {'leaf': FILL_LEAF, 'non-leaf': FILL_NONLEAF, 'unknown': FILL_UNKNOWN,
             'deprecated': FILL_DEPRECATED, 'category': FILL_CATEGORY}

FIXED_COLS = ['Dataset', 'Total Samples', 'Vulnerable', 'Safe', 'Unique CWEs']
N_FIXED    = len(FIXED_COLS)

def _write_data_sheet(ws, dataset_names, all_stats):
    global_c = Counter()
    for dn in dataset_names:
        global_c.update(all_stats[dn]['cwe_counts'])
    cwes = [cwe for cwe, _ in global_c.most_common()]

    for ci, col in enumerate(FIXED_COLS, 1):
        cell = ws.cell(1, ci, col)
        cell.fill, cell.font, cell.alignment, cell.border = FILL_HEADER, FONT_HEADER, ALIGN_C, BORDER

    for ci, cwe in enumerate(cwes, N_FIXED + 1):
        num  = _strip(cwe)
        name = nav.get_element_name(num)
        cell = ws.cell(1, ci, f'{cwe}\n{name}' if name != 'Unknown' else cwe)
        cell.fill, cell.font, cell.alignment, cell.border = _CWE_FILL[cwe_label(cwe)], FONT_CWE, ALIGN_C, BORDER

    for ri, dn in enumerate(dataset_names, 2):
        s = all_stats[dn]
        rf = FILL_ROW_ODD if ri % 2 == 1 else PatternFill()
        for ci, val in enumerate([dn, s['total'], s['vulnerable'], s['safe'], s['unique_cwes']], 1):
            cell = ws.cell(ri, ci, val)
            cell.font = FONT_DATA_B if ci == 1 else FONT_DATA
            cell.alignment = ALIGN_L if ci == 1 else ALIGN_C
            cell.fill, cell.border = rf, BORDER
        for ci, cwe in enumerate(cwes, N_FIXED + 1):
            cnt = s['cwe_counts'].get(cwe, 0)
            cell = ws.cell(ri, ci, cnt if cnt > 0 else None)
            cell.font, cell.alignment, cell.fill, cell.border = FONT_DATA, ALIGN_C, rf, BORDER

    ws.column_dimensions['A'].width = 22
    for c in range(2, N_FIXED + 1):
        ws.column_dimensions[get_column_letter(c)].width = 14
    for c in range(N_FIXED + 1, N_FIXED + len(cwes) + 1):
        ws.column_dimensions[get_column_letter(c)].width = 16
    ws.row_dimensions[1].height = 48
    ws.freeze_panes = 'B2'

wb = Workbook()
wb.remove(wb.active)

SHEET_MAP = {'real': 'C_Real', 'synth': 'C_Synth', 'ai': 'C_AI'}
for branch, sheet_name in SHEET_MAP.items():
    datasets = [n for n, s in stats.items() if s['branch'] == branch]
    if datasets:
        ws = wb.create_sheet(sheet_name)
        _write_data_sheet(ws, datasets, stats)

# Filters sheet
wf = wb.create_sheet('Filters')
filter_cols = ['Dataset', 'Branch', 'Language', 'Source', 'Positives', 'Negatives',
               'CWE Source', 'Filters Applied', 'Safe Type', 'Label Quality', 'Notes']
for ci, col in enumerate(filter_cols, 1):
    cell = wf.cell(1, ci, col)
    cell.fill, cell.font, cell.alignment, cell.border = FILL_HEADER, FONT_HEADER, ALIGN_C, BORDER

for ri, (ds_name, filt) in enumerate(DATASET_FILTERS.items(), 2):
    rf = FILL_ROW_ODD if ri % 2 == 1 else PatternFill()
    row_vals = [ds_name, filt['branch'], filt['language'], filt['source'],
                filt['positives'], filt['negatives'], filt['cwe_source'],
                filt['filters_applied'], filt['safe_type'], filt['label_quality'], filt['notes']]
    for ci, val in enumerate(row_vals, 1):
        cell = wf.cell(ri, ci, val)
        cell.font = FONT_DATA_B if ci == 1 else FONT_DATA
        cell.alignment = ALIGN_L
        cell.fill, cell.border = rf, BORDER

filter_widths = [22, 8, 8, 35, 45, 30, 30, 45, 14, 12, 40]
for ci, w in enumerate(filter_widths, 1):
    wf.column_dimensions[get_column_letter(ci)].width = w
wf.freeze_panes = 'B2'

# Legend sheet
wl = wb.create_sheet('Legend')
legend_rows = [
    ('Colour', 'Meaning'),
    ('Green  (#92D050)', 'Leaf CWE -- most specific; no children in MITRE tree'),
    ('Amber  (#FFC000)', 'Non-leaf CWE -- parent/intermediate node'),
    ('Gray   (#BFBFBF)', 'Unknown CWE -- ID not in cwec_latest.xml'),
    ('Pink   (#FFB3B3)', 'Deprecated CWE -- name starts with DEPRECATED:'),
    ('Purple (#D2B4DE)', 'Category/View -- MITRE organisational grouping'),
]
fills = [FILL_HEADER, FILL_LEAF, FILL_NONLEAF, FILL_UNKNOWN, FILL_DEPRECATED, FILL_CATEGORY]
for ri, (a, b) in enumerate(legend_rows, 1):
    ca, cb = wl.cell(ri, 1, a), wl.cell(ri, 2, b)
    ca.fill = fills[ri - 1]
    ca.font = FONT_HEADER if ri == 1 else FONT_CWE
    cb.font = FONT_HEADER if ri == 1 else FONT_DATA
    ca.alignment = cb.alignment = ALIGN_L
    ca.border = cb.border = BORDER
wl.column_dimensions['A'].width = 22
wl.column_dimensions['B'].width = 60

wb.save(XLSX_PATH)
print(f'Saved: {XLSX_PATH}')
print(f'Sheets: {wb.sheetnames}')

Saved: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\outputs\stage1_c_cpp_stats.xlsx
Sheets: ['C_Real', 'C_Synth', 'C_AI', 'Filters', 'Legend']


## 6. Summary

In [7]:
global_counter: Counter = Counter()
for s in stats.values():
    global_counter.update(s['cwe_counts'])
ALL_CWES = [cwe for cwe, _ in global_counter.most_common()]

by_label = {lbl: [c for c in ALL_CWES if cwe_label(c) == lbl]
            for lbl in ('leaf', 'non-leaf', 'unknown', 'deprecated', 'category')}
for lbl, cwes in by_label.items():
    print(f'  {lbl:<12}: {len(cwes):>4}  {cwes[:5]} ...')

  leaf        :   85  ['CWE-476', 'CWE-416', 'CWE-122', 'CWE-78', 'CWE-415'] ...
  non-leaf    :   90  ['CWE-125', 'CWE-119', 'CWE-787', 'CWE-20', 'CWE-190'] ...
  unknown     :    0  [] ...
  deprecated  :    2  ['CWE-1187', 'CWE-216'] ...
  category    :   15  ['CWE-399', 'CWE-264', 'CWE-189', 'CWE-310', 'CWE-17'] ...
